# VoiceGuard — Acoustic Deepfake Classifier Training (Google Colab T4 GPU)

This notebook trains the binary acoustic deepfake classifier (`AcousticDeepfakeCNN` using EfficientNet-B0) on a Google Colab T4 GPU.

### Operating Principles & Evaluation Integrity (06 §1-§2, 13 §6, 16 §1):
1. **Train in-domain, evaluate out-of-domain**:
   - Primary training corpus: Official ASVspoof 2019 LA benchmark (16 §1.1) via open Hugging Face mirror.
   - Mandatory out-of-domain test set: In-the-Wild deepfake speech (16 §1.3), strictly held out (never trained on).
   - **Zero Kaggle credentials / Zero registration wall**: All assets download via direct public HTTPS.
2. **Strict speaker disjointness**:
   - Zero speaker overlap across train, validation, and evaluation splits (`verify_speaker_disjointness`).
3. **Full Augmentation Pipeline (05 §2.4)**:
   - Waveform random gain [-6 dB, +6 dB]
   - Additive Gaussian noise [5, 25] dB SNR
   - SpecAugment frequency masking (2 bands, width 16) and time masking (2 bands, width 40).
4. **Google Drive Checkpointing (06 §6.1-§6.2)**:
   - Mounts Google Drive and persists checkpoints (`checkpoint_latest.pt` and `model_best.pt`) after every epoch.
   - Automatically resumes from Drive checkpoint if Colab disconnects.
5. **C1–C4 Comprehensive Evaluation Protocol (13 §6)**:
   - Evaluates C1 (In-Domain), C2 (Out-of-Domain), C3 (8 kHz μ-law Codec Degraded), C4 (10 dB SNR Noise Degraded).
   - Computes EER, AUC-ROC, F1, min t-DCF, ECE, Confusion Matrix, and the C1→C2 Generalization Gap.
   - Emits authentic `metrics.json`.


In [ ]:
# 1. Environment & GPU Check
!nvidia-smi

# Install required dependencies (datasets for direct streaming, timm for CNN backbone)
!pip install -q timm>=0.9.12 librosa>=0.10.1 soundfile>=0.12.1 scikit-learn>=1.4.0 datasets structlog webrtcvad-wheels matplotlib pyyaml scipy


In [ ]:
# 2. Mount Google Drive for persistent checkpointing & disconnect recovery (06 §6.1)
from pathlib import Path
import os, shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    checkpoint_dir = Path("/content/drive/MyDrive/VoiceGuard_Checkpoints")
except Exception:
    print("Not running in interactive Colab with Drive; using /content/checkpoints")
    checkpoint_dir = Path("/content/checkpoints")

checkpoint_dir.mkdir(parents=True, exist_ok=True)
print(f"[READY] Persistent checkpoint directory: {checkpoint_dir}")


In [ ]:
# 3. Clone VoiceGuard Repository & Configure Environment
import sys
from pathlib import Path

repo_root = Path("/content/VoiceGuard").resolve()
if not repo_root.exists():
    print("Cloning VoiceGuard repository...")
    !git clone https://github.com/AS24xADITYA/VoiceGuard.git /content/VoiceGuard
    repo_root = Path("/content/VoiceGuard").resolve()
else:
    print("VoiceGuard repository already present. Pulling latest updates...")
    !git -C /content/VoiceGuard pull

backend_dir = repo_root / "backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))
scripts_dir = repo_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))
print(f"[READY] Backend loaded from: {backend_dir}")


## 4. Real Dataset Acquisition & Preprocessing (Zero Authentication Barriers)

Per specifications (16 §1):
- **In-the-Wild**: Out-of-domain evaluation corpus (16 §1.3), downloaded directly from Hugging Face via `wget` (no login, no API token, no 403 error).
- **ASVspoof 2019 LA**: Official primary benchmark corpus (16 §1.1), downloaded via open Hugging Face dataset without registration wait.

In [ ]:
# 4. Direct Dataset Download (No Kaggle, No 403 Errors)
from pathlib import Path
import os, shutil

data_dir = Path("/content/data")
data_dir.mkdir(parents=True, exist_ok=True)
itw_dir = data_dir / "in_the_wild"

# ── [1/2] Out-of-Domain Generalization Corpus (In-the-Wild - 16 §1.3) ────────
print("--- [1/2] Downloading In-the-Wild Out-of-Domain Corpus from Hugging Face ---")
itw_zip = data_dir / "release_in_the_wild.zip"
if not itw_zip.is_file():
    !wget -c "https://huggingface.co/datasets/mueller91/In-The-Wild/resolve/main/release_in_the_wild.zip" -O "{itw_zip}"

if not itw_dir.is_dir() or not any(itw_dir.iterdir()):
    print("Extracting In-the-Wild archive...")
    !unzip -q -o "{itw_zip}" -d "{itw_dir}"
    print("[SUCCESS] In-the-Wild extracted.")

print("Preprocessing In-the-Wild evaluation set...")
!python /content/VoiceGuard/scripts/prepare_datasets.py \
    --audio-dir "{itw_dir}" \
    --output-dir /content/processed/in_the_wild \
    --dataset-type in_the_wild \
    --precompute-specs

# ── [2/2] Primary Training Corpus (ASVspoof 2019 LA - 16 §1.1) ─────────────
print("\n--- [2/2] Preparing Official ASVspoof 2019 LA from Hugging Face ---")
!python /content/VoiceGuard/scripts/prepare_datasets.py \
    --output-dir /content/processed/primary \
    --dataset-type hf_asvspoof \
    --precompute-specs


In [ ]:
# 5. Launch Acoustic CNN Training Loop with Google Drive Checkpointing (06 §6)
import json
from pathlib import Path
import shutil
import torch
from ai.acoustic.train import run_training

primary_manifest_file = Path("/content/processed/primary/manifest.json")
if not primary_manifest_file.exists():
    raise FileNotFoundError(f"Primary manifest not found at {primary_manifest_file}")

with open(primary_manifest_file, "r", encoding="utf-8") as f:
    all_primary = json.load(f)

# Partition primary into train and dev splits
train_manifest = [m for m in all_primary if m.get("split") == "train"]
dev_manifest = [m for m in all_primary if m.get("split") == "dev"]

# If dataset had no separate dev split, allocate 80/20 speaker-disjointly
if not dev_manifest:
    spk_groups = {}
    for m in all_primary:
        spk_groups.setdefault(m["speaker_id"], []).append(m)
    all_spks = sorted(list(spk_groups.keys()))
    split_idx = int(0.8 * len(all_spks))
    train_spks = set(all_spks[:split_idx])
    train_manifest = [m for m in all_primary if m["speaker_id"] in train_spks]
    dev_manifest = [m for m in all_primary if m["speaker_id"] not in train_spks]

print(f"Primary Dataset: {len(train_manifest)} train samples, {len(dev_manifest)} dev (C1) samples")

# Check for existing checkpoint to resume in case of session reset
latest_ckpt = checkpoint_dir / "checkpoint_latest.pt"
resume_path = str(latest_ckpt) if latest_ckpt.is_file() else None
if resume_path:
    print(f"★ Resuming training from persistent checkpoint: {resume_path}")

training_summary = run_training(
    train_manifest=train_manifest,
    val_manifest=dev_manifest,
    output_dir="/content/model_output",
    backbone="efficientnet_b0",
    epochs=25,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-4,
    patience=6,
    resume_checkpoint=resume_path,
)

# Synchronize best model and latest checkpoint to Google Drive
best_local = Path("/content/model_output/model_best.pt")
latest_local = Path("/content/model_output/checkpoint_latest.pt")
if best_local.is_file():
    shutil.copy(best_local, checkpoint_dir / "model_best.pt")
    shutil.copy(best_local, checkpoint_dir / "acoustic.pth")
if latest_local.is_file():
    shutil.copy(latest_local, checkpoint_dir / "checkpoint_latest.pt")
print(f"[SAVED] Checkpoints synchronized to Google Drive: {checkpoint_dir}")


In [ ]:
# 6. Comprehensive Acoustic Model Evaluation Protocol (13 §6)
import json
from pathlib import Path
import shutil
import torch
from ai.evaluation.run_eval import run_full_evaluation

# Save dev split as C1 manifest
c1_manifest_path = Path("/content/processed/primary/c1_dev_manifest.json")
with open(c1_manifest_path, "w", encoding="utf-8") as f:
    json.dump(dev_manifest, f, indent=2)

c2_manifest_path = Path("/content/processed/in_the_wild/manifest.json")
best_model_path = Path("/content/model_output/model_best.pt")
metrics_output_path = Path("/content/model_output/metrics.json")

print("Executing C1-C4 Evaluation Suite...")
eval_results = run_full_evaluation(
    checkpoint_path=best_model_path,
    c1_manifest_path=c1_manifest_path,
    c2_manifest_path=c2_manifest_path,
    output_metrics_path=metrics_output_path,
    backbone="efficientnet_b0",
    device_str="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=32,
    c4_snr_db=10.0,
)

# Copy metrics.json to persistent Google Drive
shutil.copy(metrics_output_path, checkpoint_dir / "metrics.json")
print(f"[SAVED] Metrics saved to Google Drive: {checkpoint_dir / 'metrics.json'}")


In [ ]:
# 7. Export Model Artifact & 1-Click Browser Download
import shutil
from pathlib import Path

best_model_pt = Path("/content/model_output/model_best.pt")
acoustic_pth = Path("/content/model_output/acoustic.pth")
metrics_json = Path("/content/model_output/metrics.json")

if best_model_pt.exists():
    shutil.copy(best_model_pt, acoustic_pth)
    print(f"[READY] Ready for deployment: {acoustic_pth} and {metrics_json}")
    try:
        from google.colab import files
        print("Downloading acoustic.pth...")
        files.download(str(acoustic_pth))
        print("Downloading metrics.json...")
        files.download(str(metrics_json))
        print("\n>>> Place these files in your local VoiceGuard repository at:")
        print("    VoiceGuard/backend/models/acoustic.pth")
        print("    VoiceGuard/backend/models/metrics.json")
    except Exception as e:
        print(f"Files are available in Google Drive at: {checkpoint_dir}")
else:
    print("Model checkpoint not found. Ensure training completed.")
